In [ ]:
import sys; sys.path.append('..'); sys.path.append('../..')

In [ ]:
from periodic_simulation_setup import *

In [ ]:
h = 10
w = 10 - 4.6 * 2
res = 100

triArea = h * w / res
avg_len = triArea

In [ ]:
np.linspace(0.1, 1, 10)

In [ ]:
avg_len = 0.25

In [ ]:
ipu, m, marker = periodic_unit_helper.get_parallel_tube_periodic(h, w, avg_len)

In [ ]:
m.save("mesh.obj")

In [ ]:
fuse_boundary = True

# Fuse boundary
if fuse_boundary:
    bbox = igl.bounding_box(m.vertices())

    max_x = max(bbox[0][:, 0])
    min_x = min(bbox[0][:, 0])
    max_y = max(bbox[0][:, 1])
    min_y = min(bbox[0][:, 1])

    vxs = m.vertices()
    for i, vx in enumerate(m.vertices()):
        if np.abs(vx[0] - max_x) < 1e-6:
            marker[i] = True
        if np.abs(vx[0] - min_x) < 1e-6:
            marker[i] = True    
        if np.abs(vx[1] - max_y) < 1e-6:
            marker[i] = True
        if np.abs(vx[1] - min_y) < 1e-6:
            marker[i] = True    


In [ ]:
visualization.plot_2d_mesh(m, pointList=marker, width=5, height=5)

In [ ]:
ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = marker, epsilon = 1e-9)


In [ ]:
isheet = inflation.InflatableSheet(m, marker)

In [ ]:
from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(isheet, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

In [ ]:
fixedVars, hessianShift = [], 1e-6



In [ ]:
import time, vis
benchmark.reset()
isheet.setUseTensionFieldEnergy(True)
isheet.setUseHessianProjectedEnergy(False)
isheet.pressure = 0.01
opts.niter = 2000
framerate = 1 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update()
cr = inflation.inflation_newton(isheet, fixedVars, opts, callback=cb, hessianShift = hessianShift)
benchmark.report()

In [ ]:
lambdas, modes = compute_vibrational_modes.compute_vibrational_modes(ModalAnalysisWrapper(isheet), mtype=compute_vibrational_modes.MassMatrixType.FULL, n=32, sigma=1e-3, fixedVars = fixedVars)


In [ ]:

import mode_viewer, importlib
mview = mode_viewer.ModeViewer(isheet, modes, lambdas, amplitude=50)
mview.show()

In [ ]:
# V = np.array(m.vertices())
# V[:, [0, 1]] = V[:, [1, 0]]
# F = m.elements()
# fusedVertices = [1 if vx == True else 0 for vx in marker]

In [ ]:
# import json
# with open('../data/ZigZag_3_4/ZigZag_3_4_iter_-1.json', 'w') as f:
#     data = {"Vertices": V.tolist(), "Faces": F.tolist(), "FusedVertices": fusedVertices}
#     json.dump(data, f)

In [ ]:
# visualization.plot_2d_mesh(m, pointList=marker, width=10, height=10)

In [ ]:
# with open('../data/ZigZag_3_4/ZigZag_3_4_iter_{}.json'.format(0), 'r') as f:
#     data = json.load(f)
# fusedVertices = data['FusedVertices']
# fusedVertices = [True if vx == 1 else False for vx in fusedVertices]
# V = data['Vertices']
# F = data['Faces']
# m = MeshFEM.Mesh(V, F)

In [ ]:
# visualization.plot_2d_mesh(m, pointList=fusedVertices, width=5, height=5)

In [ ]:
allowBending = False
pressure = 0.01

In [ ]:
from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)
viewer.show()

In [ ]:

framerate = 1 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])

In [ ]:
# Choose strategy for constraining rigid motion
fixedVars, hessianShift = periodic_unit_helper.get_center_fixedVars(ipu), 0
if not allowBending:
    fixedVars, hessianShift = [ipu.numVars() - 2, ipu.numVars() - 1], 1e-6
else:
    fixedVars, hessianShift = [], 1e-6

ipu.sheet.setUseTensionFieldEnergy(True)
ipu.sheet.setUseHessianProjectedEnergy(False)

ipu.sheet.pressure = pressure

benchmark.reset()

opts.niter = 100
cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)
viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])

benchmark.report()

In [ ]:
# name = 'parallel_tube'
# time_stamp = time.strftime("%Y_%m_%d_%H_%M")
# result_folder = 'output/{}/{}'.format(name, time_stamp)
# if not os.path.exists(result_folder):
#     os.makedirs(result_folder)  

In [ ]:
az_ipu = get_az_ipu_from_ipu(ipu, m, marker, True)
fixedVars, hessianShift = [az_ipu.numVars() - 2, az_ipu.numVars() - 1], 1e-6
opts.niter = 1000

In [ ]:

from tri_mesh_viewer import TriMeshViewer
az_viewer = TriMeshViewer(az_ipu, width=768, height=640)
az_viewer.showWireframe(True)

In [ ]:
az_viewer.show()

In [ ]:
az_viewer.update(scalarField=utils.getStrains(az_ipu.ipu.sheet)[:, 0])

In [ ]:
# curr_vars = az_ipu.getVars()
# curr_vars[-2] = 0.
# curr_vars[-1] = np.pi / 2
# az_ipu.setVars(curr_vars)

In [ ]:
def az_cb(it):
    framerate = 1
    if it % framerate == 0:
        az_viewer.update(scalarField=utils.getStrains(az_ipu.ipu.sheet)[:, 0])

In [ ]:
fixedVars

In [ ]:
az_optimizer = inflation.get_inflation_optimizer(az_ipu, fixedVars, opts, callback=az_cb, hessianShift = hessianShift)
cr = az_optimizer.optimize()

In [ ]:
benchmark.reset()
stiffness_shift = 1e-15
while True:
    try:
        stiffness_values, sampled_alphas = visualize_sampled_bending_stiffness(az_ipu, 1000, az_optimizer, hessianShift = stiffness_shift, fixedVars = [])
        break
    except:
        print("failed to compute stiffness with shift ", stiffness_shift)
    stiffness_shift *= 10
benchmark.report()
min(stiffness_values), max(stiffness_values)

In [ ]:
benchmark.reset()
stiffness_values, sampled_alphas = visualize_sampled_bending_stiffness(az_ipu, 100, az_optimizer, hessianShift = 1e-10, fixedVars = [], filename = "{}/stiffness_parallel_tube.png".format(result_folder))
benchmark.report()
min(stiffness_values), max(stiffness_values)

In [ ]:
(99.40110831899777 - 99.40110857789945) / 99.40110857789945

In [ ]:
benchmark.reset()
stiffness_values, sampled_alphas = visualize_sampled_bending_stiffness(az_ipu, 100, az_optimizer, hessianShift = 1e-10, fixedVars = [], filename = "{}/stiffness_parallel_tube.png".format(result_folder), use_bases=False)
benchmark.report()
min(stiffness_values), max(stiffness_values)

In [ ]:
min(stiffness_values)

In [ ]:
points = visualize_average_deformation_gradient(ipu, 100, filename = "{}/average_deformation_gradient_parallel_tube.png".format(result_folder))

In [ ]:

an = np.linspace(0, 2 * np.pi, 100)
fig, ax = plt.subplots(1, 1)
ax.plot(np.cos(an), np.sin(an))

ax.plot(points[:, 0], points[:, 1])

ax.set_aspect('equal', 'box')
ax.set_title('still a circle, auto-adjusted data limits', fontsize=10)

fig.tight_layout()

plt.show()